<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/FOXO_ALPHAZ_POV_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foxo alphaz POV — gay621 A100

Ram-horn orange fox, collar, shorts yanked down, POV pounce. **Not** the Foxo.png bodysuit lock.

| | |
|---|---|
| GPU | Runtime → **A100** + High-RAM |
| Weights | Drive `MyDrive/models/gay621FurryMaleFocus_gay621XLV10.safetensors` (6938040736 bytes). Do not re-upload. |
| OUT | Drive `MyDrive/AI-outputs/foxo_alphaz_pov/` |

This is **not** the Comfy last-cell job hopper and **not** Flask Bridge v2. Same checkpoint. Prompt is in the last cell. Re-run the last cell only for more seeds.

Connect A100 → Run all → authorize Drive. First load ~30s after the copy.


In [ ]:
# 0) GPU gate
import torch, sys
print("python", sys.version.split()[0])
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram:.1f} GB")
if vram < 15:
    raise SystemExit(
        f"Need a real GPU. This box is {vram:.1f}GB ({name}). "
        "Runtime → Change runtime type → A100."
    )
if vram < 24:
    print("WARN: under 24GB. SDXL will run; A100 is the intended runtime.")
else:
    print("A100-class OK.")


In [ ]:
# 1) Drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
for p in (DRIVE / "models", DRIVE / "AI-outputs" / "foxo_alphaz_pov"):
    p.mkdir(parents=True, exist_ok=True)
    print(p, "ok")


In [ ]:
# 2) Deps
!pip install -q diffusers transformers accelerate safetensors pillow
print("deps ok")


In [ ]:
# 3) Pin gay621. Size-check. Do not glob. Do not re-download.
from pathlib import Path

NEED = 6938040736
CKPT = "gay621FurryMaleFocus_gay621XLV10.safetensors"
SRC = Path("/content/drive/MyDrive/models") / CKPT
if not SRC.is_file():
    raise FileNotFoundError("Missing %s. Stop. Do not download a substitute." % SRC)
src_sz = SRC.stat().st_size
print("Drive ckpt bytes:", src_sz)
if src_sz != NEED:
    print("WARN: Drive size is not the known 6938040736. Loading anyway.")
DRIVE_MODEL_PATH = str(SRC)
print("DRIVE_MODEL_PATH", DRIVE_MODEL_PATH)


In [ ]:
# 4) Load pipeline
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

print("Loading gay621 SDXL from Drive...")
pipe = StableDiffusionXLPipeline.from_single_file(
    DRIVE_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers on")
except Exception as e:
    print("xformers skipped:", type(e).__name__, e)
print("Pipeline ready on", torch.cuda.get_device_name(0))


In [ ]:
# 5) GENERATE — re-run THIS cell for more seeds. Bump COUNT or SEED0.
import os, json, time, torch
from pathlib import Path
from datetime import datetime, timezone

COUNT = 12
SEED0 = 270913
WIDTH, HEIGHT = 832, 1216
STEPS = 30
CFG = 7.0
PREFIX = "foxo_alphaz_pov"
OUT = Path("/content/drive/MyDrive/AI-outputs/foxo_alphaz_pov")
OUT.mkdir(parents=True, exist_ok=True)

# Sex-first, then alphaz identity. Ram horns stay. Not Foxo.png bodysuit.
PROMPT = (
    "gay, male, solo, explicit, pov, first person view, viewer being mounted, "
    "pouncing, pinning down, energetic mid-lunge, hips flush, thrusting toward camera, "
    "penetration, sinking in, dominant, hungry, overpowering, "
    "anthro fox, orange-red fur, cream chest fur, cream belly fur, "
    "darker brown torso markings, spotted thighs, striped forearms, striped lower legs, "
    "lean muscular body, large bright blue eyes, half-lidded, lust, confident playful smirk, "
    "fluffy swept-over orange hair, long animal ears, pink inner ear fur, "
    "large curved ram horns on the orange fox, black collar, aqua-blue spikes, buckle, "
    "dark charcoal athletic shorts yanked down around one thigh, drawstring undone, "
    "thick erect anthro penis, flushed tapered tip, heavy balls, precum, glistening, "
    "paw hands, paw feet, rounded toes, dark claws, not webbed, "
    "one powerful leg planted forward, other leg lifted and braced, "
    "arms reaching toward viewer, grabbing, pinning, claws catching viewer shoulders, "
    "sweat sheen on chest, inner thighs, and shaft, "
    "low angle camera, looking up along his body, intimate, "
    "dark bluish-gray studio gradient, soft lighting, warm rim light on wet fur and collar spikes, "
    "3d render, polished 3d, highly detailed fur, highly detailed anatomy, octane render, masterpiece"
)

NEG = (
    "low quality, blurry, deformed, bad anatomy, extra limbs, extra penises, "
    "watermark, text, censored, female, child, cub, underage, "
    "webbed hands, webbed feet, frog hands, duck feet, "
    "two foxes, two males, duo, bodysuit, black sleeves, black leggings, "
    "horns missing, no horns, goat, frog, hyena, "
    "cartoon, cel shading, flat color, thick outlines, chibi, sticker, "
    "clothes covering penis, shorts pulled up"
)

meta = {
    "prompt": PROMPT,
    "negative": NEG,
    "count": COUNT,
    "seed0": SEED0,
    "width": WIDTH,
    "height": HEIGHT,
    "steps": STEPS,
    "cfg": CFG,
    "ckpt": "gay621FurryMaleFocus_gay621XLV10.safetensors",
    "lock": "alphaz ram-horn fox, not Foxo.png bodysuit",
    "utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
(OUT / "prompt.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("wrote", OUT / "prompt.json")

saved = 0
skipped = 0
for i in range(COUNT):
    seed = SEED0 + i
    dest = OUT / f"{PREFIX}_{i:02d}_s{seed}.png"
    if dest.is_file() and dest.stat().st_size > 200000:
        print(f"skip exists {dest.name}")
        skipped += 1
        continue
    gen = torch.Generator("cuda").manual_seed(seed)
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(
            prompt=PROMPT,
            negative_prompt=NEG,
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=STEPS,
            guidance_scale=CFG,
            generator=gen,
        ).images[0]
    img.save(dest)
    dt = time.time() - t0
    print(f"saved {dest.name}  {dest.stat().st_size} bytes  {dt:.1f}s")
    saved += 1
    torch.cuda.empty_cache()

print(f"done. saved={saved} skipped={skipped} out={OUT}")
